# 4068. 会议收益：搜索到二分优化 DP

选择互不重叠的会议，收益为会议收入加相邻已选会议之间的空闲时长。首场之前、末场之后不计收益，首尾相接允许。

原文标题为 4068，但挂在 2916 题下；本笔记按文章实际讨论的会议问题命名，并保留原始链接。

[原文](https://leetcode.cn/problems/subarrays-distinct-element-sum-of-squares-ii/solution/4068kai-hui-zhuan-qian-mo-yu-ye-zhuan-qi-akyz/) · 整理于 2026-10-17。原代码保持原样；补充注释另放单元格。

## DFS：枚举安排

状态包含下一场搜索起点、累计收益和前次结束时间。相同结尾却收益不同会产生多个缓存键，最坏仍可能指数增长。

In [ ]:
# current_end == -1 表示尚未选会议，不能收取首场之前的空闲收入。
# 进入状态立即更新 ans，因为当前方案可以随时结束。
# for 从 meeting_id 枚举，已经包含跳过某些会议的选择。

In [ ]:
# Solution 1. DFS
from typing import List, Optional, Tuple, Dict, Set
from functools import lru_cache

class Solution:
    def maxEarnings(self, meetings: list[list[int]]) -> int:
        orderd_meetings = sorted(meetings, key=lambda x: x[0])
        current_revenue = 0
        current_end = -1
        ans = 0
        @lru_cache(None)
        def dfs(meeting_id:int, current_revenue:int, current_end:int):
            nonlocal ans
            ans = max(ans, current_revenue)
            if meeting_id == len(orderd_meetings):
                return
            for i in range(meeting_id, len(orderd_meetings)):
                if current_end <= orderd_meetings[i][0]:
                    extra_revenue = orderd_meetings[i][0] - current_end if current_end < orderd_meetings[i][0] and current_end >= 0 else 0
                    dfs(i + 1, current_revenue + extra_revenue + orderd_meetings[i][2], orderd_meetings[i][1])
                else:
                    continue

        dfs(0, current_revenue, current_end)
        return ans

In [ ]:
assert Solution().maxEarnings([[3, 5, 4], [4, 7, 8], [8, 10, 3]]) == 12
assert Solution().maxEarnings([[1, 2, 3], [2, 4, 5]]) == 8
assert Solution().maxEarnings([[3, 8, 2]]) == 2

## BFS：改变访问顺序

队列仍保存完整方案状态，没有减少候选方案。visited 未使用；step 统计出队次数，既非层数，也非答案。最坏时间与队列空间仍可能指数增长。

In [ ]:
# 合法条件为 prev_end <= 下一场开始时间。
# 最大收益问题不能在首次到达终点时停止。
# 原代码 prev_end > 0 才计算空闲收入；若输入允许结束时间为 0，需留意边界。

In [ ]:
# Solution 2. BFS
from typing import List, Optional, Tuple, Dict, Set
from collections import deque
class Solution:
    def maxEarnings(self, meetings: list[list[int]]) -> int:
        ordered_meetings = sorted(meetings, key=lambda x: x[0])
        n = len(ordered_meetings)
        queue = deque([])
        visited = set()
        step = 0
        ans = 0
        # (下一场从哪个下标开始找, 当前总收益, 上一场结束时间)
        queue.append((0,0,-1))

        while queue:
            cur = queue.popleft()
            meeting_id, current_revenue, prev_end = cur[0], cur[1], cur[2]
            ans = max(ans, current_revenue)
            for i in range(meeting_id,n):
                if prev_end <= ordered_meetings[i][0]:
                    extra_revenue = (ordered_meetings[i][0] - prev_end) if prev_end>0 else 0
                    next_revenue = extra_revenue + ordered_meetings[i][2] + current_revenue

                    queue.append(
                        (i+1,
                        next_revenue,
                        ordered_meetings[i][1]
                        )
                    )
                else:
                    continue
            step+=1

        return ans

In [ ]:
assert Solution().maxEarnings([[3, 5, 4], [4, 7, 8], [8, 10, 3]]) == 12
assert Solution().maxEarnings([[1, 2, 3], [2, 4, 5]]) == 8
assert Solution().maxEarnings([[3, 8, 2]]) == 2

## 基础 DP：合并同结尾方案

`dp[i]` 表示最后参加会议 i 时的最大收益。同结尾方案后续选择和增量相同，所以只保留当前最优收益即可。时间 O(n²)，空间 O(n)。

In [ ]:
# 每个状态先取只参加自己的收益。
# 合法前驱满足 end[j] <= start[i]。
# 转移为 dp[j] + start[i] - end[j] + revenue[i]。
# 答案取 max(dp)，最佳方案不必以排序后的末场结束。

In [ ]:
# Solution 3. DP
from typing import List, Optional, Tuple, Dict, Set

class Solution:
    def maxEarnings(self, meetings: list[list[int]]) -> int:
        ordered_meetings = sorted(meetings, key=lambda x: x[0])
        n = len(ordered_meetings)
        dp = [0] * n
        for i in range(n):
            rev = ordered_meetings[i][2]
            for j in range(i):
                if ordered_meetings[j][1]>ordered_meetings[i][0]:
                    continue
                else:
                    extra_revenue = ordered_meetings[i][0] - ordered_meetings[j][1]
                    cur = dp[j]+extra_revenue + ordered_meetings[i][2]
                rev = max(rev,cur)
            dp[i] = rev
        return max(dp)

In [ ]:
assert Solution().maxEarnings([[3, 5, 4], [4, 7, 8], [8, 10, 3]]) == 12
assert Solution().maxEarnings([[1, 2, 3], [2, 4, 5]]) == 8
assert Solution().maxEarnings([[3, 8, 2]]) == 2

## 优化 DP：前缀最优值与二分

拆分转移为 `(dp[j]-end[j]) + (start[i]+revenue[i])`。按 end 排序，合法前驱组成前缀；二分找到范围，前缀最大值给出最优前驱分数。

In [ ]:
# bisect_right 包含 end[j] == start[i]；查找范围 [0,i) 仅含已处理会议。
# prefix_best[i] = max(dp[j]-end[j])，其中 0 <= j <= i。
# 真实分数可以为负，不能把未写入的占位 0 当成合法前驱。
# 输入按原题要求非空；原代码没有处理空会议列表。

In [ ]:
# Solution 4. DP OPT1

'''
dp[i] 表示最后选择第 i 场会议时的最大总收益。
接上当前会议后：
空闲收入 = start[i] - end[j]
会议收入 = revenue[i] + dp[j]
总收益 = (dp[j] - end[j]) + (revenue[i] + start[i])
因此，用 prefix_best 保存前缀中最大的 dp[j] - end[j]。
'''

from typing import List, Optional, Tuple, Dict, Set
from bisect import bisect_right

class Solution:
    def maxEarnings(self, meetings: list[list[int]]) -> int:
        ordered_meetings = sorted(meetings,key=lambda x:x[1])
        ends = [x[1] for x in ordered_meetings]
        n = len(ordered_meetings)
        dp = [0] * n
        prefix_best = [0] * n
        for i in range(n):
            # 先考虑只参加当前会议
            rev = ordered_meetings[i][2]
            # 找到最后一个结束时间不晚于当前开始时间的前驱
            j = bisect_right(ends,ordered_meetings[i][0],0, i) - 1
            if j >=0:
                cur = prefix_best[j] + ordered_meetings[i][2] + ordered_meetings[i][0]
                rev = max(cur,rev)
            dp[i] = rev

            # 当前会议自己的前驱分数
            prefix_best[i] = dp[i] - ordered_meetings[i][1]

            # 合并当前值和前一个前缀的最大值
            if i>0:
                prefix_best[i] = max(prefix_best[i],prefix_best[i-1])

        return max(dp)

In [ ]:
assert Solution().maxEarnings([[3, 5, 4], [4, 7, 8], [8, 10, 3]]) == 12
assert Solution().maxEarnings([[1, 2, 3], [2, 4, 5]]) == 8
assert Solution().maxEarnings([[3, 8, 2]]) == 2

DFS/BFS 的关键瓶颈是方案数量；基础 DP 压缩为 n 个状态后，瓶颈变成逐个扫描前驱。最终版排序与 n 次二分共 O(n log n)，空间 O(n)。